In [2]:
print("importing libraries...")
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch
import os
import accelerate
import torch.distributed as dist
import tqdm

c:\Users\jorin\.conda\envs\AGI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# params

model_name = "google/gemma-2-2b-it"
dataset_name, split_name = "jeggers/CoT-Collection", "test_in_dist"
question_column_name = "final_input"
answer_column_name = "final_target"
max_gen_length = 380
batch_size = 128
pad_to_multiple_of = 8
cot_trigger = "BOT: "
answer_trigger = "ANSWER: "
instruction = ""
format_input = lambda x: f"{instruction}{x[question_column_name]}\n{cot_trigger}"

In [7]:

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
# explicitly set pad/eos token????????
######################################

print("load dataset...")
df = load_dataset(dataset_name, split=split_name).to_pandas()
df["formatted_input"] = df.apply(format_input, axis=1)
df["group"] = df.index // batch_size
    
def tokenize_input(batch):
    res = tokenizer(
        batch["formatted_input"].tolist(),
        return_tensors="pt",
        pad_to_multiple_of=pad_to_multiple_of,
        padding=True,
        truncation=False,
    )
    res["targets"] = batch[answer_column_name].tolist()
    return res

print("tokenize dataset...")
batches = df.groupby("group").apply(tokenize_input)

group
0     [input_ids, attention_mask]
1     [input_ids, attention_mask]
2     [input_ids, attention_mask]
3     [input_ids, attention_mask]
4     [input_ids, attention_mask]
5     [input_ids, attention_mask]
6     [input_ids, attention_mask]
7     [input_ids, attention_mask]
8     [input_ids, attention_mask]
9     [input_ids, attention_mask]
10    [input_ids, attention_mask]
11    [input_ids, attention_mask]
12    [input_ids, attention_mask]
13    [input_ids, attention_mask]
14    [input_ids, attention_mask]
15    [input_ids, attention_mask]
dtype: object

In [1]:
print("load model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2"
)
model.eval()

print("move model to multi GPUs...")
distributed_state = accelerate.PartialState()
model.to(distributed_state.device)


NameError: name 'dataset' is not defined

In [ ]:
print("evaluating...")
results = []
with distributed_state.split_between_processes(batches.tolist()) as batches:
    for batch in tqdm(batches, disable=not accelerate.is_main_process()):
        with torch.no_grad():
            outputs = model.generate(
                **batch,
                max_new_tokens=max_gen_length,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                temperature=0.0,
                num_return_sequences=1,
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(list(zip(decoded, batch["targets"])))

In [11]:

# takes a batch strings that contain input and completion
# returns a list of completion strings
def extract_completion_batch(input_and_completion_batch):
    cot_trigger_count_in_instructions = instruction.count(cot_trigger)
    splitted = [res.split(cot_trigger) for res in input_and_completion_batch]
    return [
        cot_trigger.join(split[cot_trigger_count_in_instructions + 1 :])
        for split in splitted
    ]


# takes a batch strings that contain only the completion
# returns a list of answer strings (part after the first found answer trigger)
# returns empty string if no answer trigger is found
def extract_answer_cot_batch(answer_cot_batch):
    splitted_batch = [res.split(answer_trigger) for res in answer_cot_batch]
    return [
        answer_trigger.join(splitted[1:]) if len(splitted) >= 2 else ""
        for splitted in splitted_batch
    ]

def check_equal(a, b):
    return a.lower().strip() == b.lower().strip()

In [ ]:
if distributed_state.is_main_process:
    gathered_results = accelerate.utils.gather_object(results)
    gathered_results = list(zip(*gathered_results))

    # cut if apply_padding = True
    # gathered_results = gathered_results[:len(df)]

    completions = extract_completion_batch(gathered_results[0])
    answers = extract_answer_cot_batch(completions)
    targets = gathered_results[1]
    correct = [check_equal(a, b) for a, b in zip(answers, targets)]
    accuracy = sum(correct) / len(correct)
    print(f"Accuracy: {accuracy:.2f}")

        